# Anatomía de una Falla · Parte II — 01 · Análisis exploratorio (EDA)

Este notebook recorre el dataset **NASA N-CMAPSS** antes de modelar. Cada sección explica **qué se mira y por qué**;
los números, tablas y gráficos se calculan desde los datos en `src/ncmapss/eda.py` (etapa `eda` de `scripts/run_all.py`)
y se leen de `outputs/<modo>/eda.json`. No hay valores escritos a mano: si cambian los datos o la configuración,
cambian las lecturas.

**Cómo regenerarlo:** `python scripts/run_all.py --only prepare health residuals features eda` (o `--sample` para la versión rápida).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from src.ncmapss import nb
MODE = "full"          # "sample" si corriste el pipeline con --sample

In [2]:
# Opcional: recalcular el EDA desde los datos procesados (unos segundos)
# from src.ncmapss import eda; from src.ncmapss.config import load_config, get_paths
# eda.run(load_config(), get_paths(load_config(), MODE))
E = nb.load("eda", MODE)
print("EDA generado el", E["generated"], "· modo", E["mode"])
nb.facts()

EDA generado el 21/10/2026 08:40 · modo full


,valor
med_r2,"99,998 %"
corr_strong,118 de 496
norm_rho_raw,"-0,42"
norm_rho_res,"-0,99"
n_profiles,554
profile_eta_t48,25 %
t48_within,±123 °R
t48_drift,+29 °R
n_combos,7
frac_sano,31 %


## 1 · Estructura del dataset

N-CMAPSS tiene **un registro por segundo de vuelo**. Cada archivo `N-CMAPSS_DSxx.h5` es un *escenario de falla*
(qué componentes se degradan) y trae dos particiones: **desarrollo** (para entrenar) y **prueba** (motores nuevos).
Las variables vienen en seis bloques:

| bloque | contenido | ¿existe en un avión real? |
|---|---|---|
| `W` | condiciones de vuelo: altitud, Mach, potencia (TRA), temperatura de entrada (T2) | sí |
| `X_s` | 14 sensores físicos: temperaturas, presiones, giros, combustible | sí |
| `X_v` | 14 sensores virtuales del simulador | no |
| `T` | 10 parámetros de salud (eficiencia y flujo de cada componente) | no |
| `Y` | RUL: vuelos que le quedan al motor | es lo que se predice |
| `A` | motor, n° de vuelo, clase de vuelo (Fc) y estado de salud (hs) | sí (salvo hs) |

In [3]:
nb.table('sizes')

,dataset,partición,motores,vuelos,registros (1 Hz)
0,DS01,dev,6,553,4.906.636
1,DS01,test,4,341,2.735.232
2,DS02,dev,6,446,5.263.447
3,DS02,test,3,202,1.253.743
4,DS03,dev,9,663,5.571.277
5,DS03,test,6,438,4.251.560
6,DS04,dev,6,512,6.377.452
7,DS04,test,4,344,3.602.561
8,DS05,dev,6,491,4.350.606
9,DS05,test,4,327,2.562.046


In [4]:
nb.show('sizes')

> **Lectura.** **69,9 millones de segundos** de vuelo (≈ 19.417 h); el 63 % es de desarrollo y el subconjunto más grande es **DS04**. **DS08d** viene truncado en Kaggle y quedó afuera.

### Calidad de los datos: faltantes, infinitos y consistencia

Se revisan **todos los segundos** (no una muestra): nulos, infinitos, rangos físicos y reglas que el dataset debería
cumplir (RUL = vida total − n° de vuelo, vuelos numerados sin huecos, clase de vuelo fija por motor, estado de salud
que nunca vuelve a "sano"). También se chequea la integridad de cada archivo: uno de los `.h5` de Kaggle está truncado.

In [5]:
nb.checks()

,verificación,resultado,detalle
0,Archivos íntegros,✘,9 de 10 archivos se abren completos; DS08d est...
1,Valores faltantes,✔,0 nulos en 3.005.712.943 celdas revisadas (tod...
2,Valores infinitos,✔,0 valores infinitos.
3,Vuelos sin saltos ni repetidos,✔,"Cada motor tiene sus vuelos numerados 1, 2, 3…..."
4,RUL coherente,✔,"RUL = vida total − n° de vuelo en el 100,0 % d..."
5,Clase de vuelo fija por motor,✔,Cada motor vuela siempre la misma clase (corta...
6,Estado de salud consistente,✔,"hs es constante dentro de cada vuelo y, una ve..."
7,Rangos físicos,✔,"Todas las temperaturas, presiones, giros y flu..."


In [6]:
nb.table('missing')

,variable,registros,nulos,infinitos,mínimo,máximo
0,alt,69.900.301,0,0,"3.001,000","35.033,000"
1,Mach,69.900.301,0,0,"0,000","0,749"
2,TRA,69.900.301,0,0,"23,555","88,769"
3,T2,69.900.301,0,0,"421,378","534,383"
4,T24,69.900.301,0,0,"483,350","637,092"
5,T30,69.900.301,0,0,"1.060,970","1.547,957"
6,T48,69.900.301,0,0,"916,706","2.006,065"
7,T50,69.900.301,0,0,"676,966","1.374,802"
8,P15,69.900.301,0,0,"5,816","20,451"
9,P2,69.900.301,0,0,"4,372","15,700"


La **variación relativa** (desvío / media) muestra cuánto se mueve cada variable. Que varíe poco no significa que no sirva.

In [7]:
nb.show('quality')

> **Lectura.** Las variables que menos varían (**Nc, T24, T2**) no son inútiles: T48 varía sólo un 7,4 % y, como se ve más adelante, es uno de los mejores indicadores del desgaste. La que más varía es **alt** (49 %).

## 2 · Perfiles de vuelo y condiciones operativas

Cada ciclo es un **vuelo completo** con condiciones que cambian segundo a segundo. Los motores se agrupan en tres
**clases de vuelo**: corto (Fc = 1), mediano (Fc = 2) y largo (Fc = 3). Miramos un vuelo, la envolvente de operación,
la duración de cada clase y la forma típica de cada perfil.

In [8]:
nb.show('profile')

> **Lectura.** Vuelo de **196 minutos**. T48 sigue a la potencia (r = 0,84 con TRA): lo que más mueve a los sensores es cómo se vuela. Los registros no incluyen pista ni rodaje: empiezan, en la mediana de todos los vuelos, a 3.002 ft.

In [9]:
nb.show('envelope')

> **Lectura.** Altitud y velocidad crecen juntas (r = 0,94). Sólo el **18 %** del tiempo transcurre por encima de 25.000 ft, y las condiciones son continuas: no hay unos pocos regímenes fijos que permitan separar los datos fácilmente (a diferencia del C-MAPSS original).

In [10]:
nb.show('flight_class')
nb.table('flight_class')

> **Lectura.** Duración media de **1,2 h** (corto), **2,5 h** (mediano), **3,9 h** (largo). Los vuelos largos además suben más (26.808 ft contra 12.558 ft de altitud máxima media). Cada motor hace siempre la misma clase de vuelo, así que la clase es una propiedad del motor y no del vuelo.

,clase,motores,vuelos,duración media [h],altitud máx. media [ft],Mach máx. medio
0,1 · corto,27,2.316,"1,20",12.558,"0,535"
1,2 · mediano,33,2.436,"2,46",21.341,"0,635"
2,3 · largo,39,2.721,"3,91",26.808,"0,696"


In [11]:
nb.show('flight_shapes')

> **Lectura.** Las tres clases siguen la secuencia ascenso → crucero → descenso, pero con regímenes distintos: la fracción del vuelo cerca de la altitud máxima (≥ 90 %) es **35 %** en los cortos, **34 %** en los medianos, **40 %** en los largos. Esa diferencia de régimen operativo es la que la Línea 2 propone separar del desgaste.

## 3 · Contexto: separar el vuelo del desgaste

La pregunta clave: ¿cuánto de lo que mide un sensor depende de **cómo se vuela** y cuánto del **desgaste**?
Se entrena un *modelo de motor sano* (red neuronal chica) que predice cada sensor desde las 4 condiciones de vuelo,
usando sólo vuelos sanos de los motores de entrenamiento. Lo medido menos lo esperado es el **residuo**.

In [12]:
nb.show('context')

> **Lectura.** T48 depende sobre todo de la potencia (**r = 0,82**). A igual potencia, cada 10.000 ft de altura la bajan unos **102 °R**. Dos vuelos distintos no se pueden comparar directamente.

In [13]:
nb.show('variance_explained')

> **Lectura.** Con sólo las cuatro condiciones de vuelo se explica el **99,998 %** de un sensor físico típico (mediana, en motores sanos que el modelo no vio). En T48 queda sin explicar un **0,016 %**: ahí adentro está el desgaste. Lo que más queda sin explicar son **SmLPC, SmHPC, T50**.

In [14]:
nb.show('corr')

> **Lectura.** **118 de 496** pares tienen |r| > 0,95. Se forman bloques de variables casi redundantes (por ejemplo W25–W32 r = 1,00, W48–W50 r = 1,00, W22–W31 r = 1,00). Hay mucha información repetida: conviene trabajar con pocos indicadores bien elegidos o con modelos que toleren la colinealidad.

In [15]:
nb.show('normalization')

> **Lectura.** Motor 1 de DS01. Medida, T48 casi no se relaciona con la vida restante (ρ = **-0,42**); sin el efecto del vuelo aparece la curva de desgaste (ρ = **-0,99**): plana hasta el vuelo 36 y cada vez más empinada hasta la falla en el vuelo 100.

### Los perfiles de vuelo se repiten

El simulador reutiliza un conjunto finito de **vuelos reales grabados**. Eso deja en el residuo un sesgo propio de
cada perfil, que conviene conocer antes de modelar (y que es central para la Línea 2: generalizar entre tipos de vuelo).

In [16]:
nb.show('profiles')

> **Lectura.** Los 7.473 vuelos usan sólo **554 perfiles distintos** (grabaciones reales que el simulador reutiliza): 36 en vuelos cortos, 233 en vuelos medianos, 285 en vuelos largos. El 84 % de los perfiles aparece en más de un motor. En la etapa sana, el perfil explica el **25 %** de la variación del residuo de T48 entre vuelos (y más del 90 % en P15, P50, P21, Nf, P2): el residuo conserva un sesgo propio de cada perfil. Por eso el modelo promedia varios vuelos (medias móviles) y recibe descriptores del vuelo.

## 4 · Qué componentes se desgastan

Los parámetros de salud (`T`) no se miden en la realidad, pero en la simulación muestran **qué se daña** en cada
escenario y **cómo evoluciona** el daño. El estado `hs` marca cuándo empieza la degradación anormal.

In [17]:
nb.show('health_heat')
nb.table('health_heat')

> **Lectura.** Cada subconjunto daña componentes distintos: **DS01**: HPT; **DS02**: HPT + LPT; **DS03**: HPT + LPT; **DS04**: fan; **DS05**: HPC; **DS06**: LPC + HPC; **DS07**: LPT; **DS08a**: fan + LPC + HPC + HPT + LPT; **DS08c**: fan + LPC + HPC + HPT + LPT. La mayor pérdida es la de fan_eff en DS04 (7,7 %); DS08a y DS08c desgastan los cinco componentes a la vez.

,dataset,fan_eff,fan_flow,LPC_eff,LPC_flow,HPC_eff,HPC_flow,HPT_eff,HPT_flow,LPT_eff,LPT_flow
0,DS01,"0,00","0,00","0,00","0,00","0,00","0,00","1,72","0,00","0,00","0,00"
1,DS02,"0,00","0,00","0,00","0,00","0,00","0,00","1,22","0,00","0,80","0,89"
2,DS03,"0,00","0,00","0,00","0,00","0,00","0,00","0,81","0,00","1,05","1,55"
3,DS04,"7,69","5,39","0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00"
4,DS05,"0,00","0,00","0,00","0,00","1,97","3,43","0,00","0,00","0,00","0,00"
5,DS06,"0,00","0,00","3,22","3,79","1,77","3,04","0,00","0,00","0,00","0,00"
6,DS07,"0,00","0,00","0,00","0,00","0,00","0,00","0,00","0,00","2,12","2,60"
7,DS08a,"0,95","1,68","1,40","1,68","0,76","1,34","0,73","1,37","0,87","1,40"
8,DS08c,"0,99","1,20","0,91","1,00","0,68","0,93","0,74","0,81","0,86","0,81"


In [18]:
nb.show('health_traj')

> **Lectura.** Dos etapas en todos los motores: una caída lenta y casi recta (azul) y, desde un vuelo distinto para cada uno, una caída cada vez más pronunciada (naranja). Además, cada motor arranca con una salud inicial diferente: la diferencia entre motores en el primer vuelo llega al 8 % de la pérdida total del componente.

In [19]:
nb.show('hs')

> **Lectura.** Un motor está sano, en promedio, sólo el **31 %** de su vida (16 % a 50 %); la degradación anormal empieza entre el vuelo 15 y el 38. Durante la etapa sana el RUL es casi imposible de estimar con precisión: ahí deberían esperarse los intervalos más anchos.

## 5 · Evolución de los sensores y sensores más informativos

Ya sin el efecto del vuelo: qué sensores cambian al acercarse la falla (correlación de Spearman con la cercanía a la
falla), cómo evolucionan a lo largo de la vida, si hay valores atípicos y cómo se ve el estado del motor resumido en dos ejes.

In [20]:
nb.show('rul_corr')

> **Lectura.** Entre los sensores físicos, los que mejor siguen el desgaste en promedio son **T48, T50, Wf, P24**. En DS04 (sólo fan) reaccionan **SmFan, P24, W21**. T30, P15, P21, P24, Nc, Wf cambian de signo según el escenario (suben en unos y bajan en otros). Nf y P2 casi no cambian (|ρ| medio ≤ 0,04).

In [21]:
nb.show('sensor_trends')

> **Lectura.** Cerca de cero con mucha vida por delante y cada vez más lejos hacia el final. La mediana de T48 supera 2 desvíos cuando faltan **33 vuelos** y llega a z ≈ 27,1 en la falla. La banda es ancha porque cada motor parte de una salud distinta: por eso el modelado usa también el residuo respecto de los primeros vuelos de cada motor.

In [22]:
nb.show('life_evolution')

> **Lectura.** Con la vida normalizada (0 % = primer vuelo, 100 % = falla) todos los escenarios muestran la misma forma: meseta durante la primera mitad y aceleración al final. En T48, el residuo típico del último 10 % de la vida es **26,7 veces** el de la primera mitad. La forma es común, pero la magnitud y el sensor que reacciona dependen del escenario.

In [23]:
nb.show('outliers')
nb.table('outliers')

> **Lectura.** Un vuelo es extremo si algún residuo se aleja más de 5 desvíos de lo normal en un motor sano. Son el 39 % de los vuelos, pero el **99 %** está en la etapa degradada; entre los vuelos sanos sólo el 0,7 % es extremo. No son errores de medición sino desgaste (y en la etapa sana, el efecto del perfil de vuelo). Por eso **no se elimina ningún vuelo**: los extremos son justamente la señal a predecir.

,sensor,% sanos extremos,% degradados extremos,vuelos extremos
0,T48,"0,00","41,57",2156
1,T50,"0,00","40,61",2106
2,Wf,"0,00","36,95",1916
3,P40,"0,26","35,29",1836
4,P24,"0,00","34,17",1772
5,T24,"0,00","33,88",1757
6,Nc,"0,00","32,03",1661
7,Ps30,"0,26","31,51",1640
8,T30,"0,00","28,38",1472
9,P50,"0,00","18,28",948


In [24]:
nb.show('pca')

> **Lectura.** Los vuelos sanos quedan juntos en una zona chica y cada escenario se aleja en su propia dirección (las direcciones de falla abarcan 96° alrededor de la zona sana). La distancia a la zona sana tiene ρ = **-0,85** con el RUL: sirve como indicador de salud general (lo usa la Línea 3).

## 6 · Vida útil y distribución del RUL

In [25]:
nb.show('lifespan')
nb.table('lifespan')

> **Lectura.** Vidas de **48** a **100** vuelos (media 75). Las más cortas son **DS08c, DS08a** (55 y 66 vuelos de media). Como cada motor dura distinto, al evaluar hay que separar motores completos: si un motor aparece en entrenamiento y prueba, el modelo "conoce" su vida.

,dataset,media,mínimo,máximo
0,DS01,"89,4",75,100
1,DS02,"72,0",59,89
2,DS03,"73,4",59,93
3,DS04,"85,6",69,100
4,DS05,"81,8",65,100
5,DS06,"79,7",65,99
6,DS07,"81,2",67,96
7,DS08a,"66,3",51,95
8,DS08c,"55,3",48,62


In [26]:
nb.show('life_vs_fc')

> **Lectura.** Vuelos cortos: **86 vuelos** de vida media; largos: **70**. En horas pasa lo contrario (103 h contra 273 h). Es una asociación (la clase se mezcla con el escenario), pero la clase de vuelo pesa en el RUL y se incluye como variable del modelo.

In [27]:
nb.show('rul_dist')

> **Lectura.** Cada motor aporta un vuelo por cada valor de RUL, así que el histograma es plano hasta **47** y luego baja (máx. 99). Mediana: 37 vuelos. Hay pocos ejemplos de motores con mucha vida por delante: es donde el modelo tendrá menos datos y donde la incertidumbre debería ser mayor.

## 7 · Conclusiones del EDA → decisiones de modelado

Cada decisión del modelado (notebook `02_rul_incertidumbre.ipynb`) responde a un hallazgo de este análisis:

In [28]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
nb.decisions()

,hallazgo del EDA,decisión de modelado
0,"Las condiciones de vuelo explican el 99,998 % de cada sensor.",Trabajar con residuos (medido − motor sano) en lugar de los sensores crudos.
1,Hay sólo 554 perfiles de vuelo reutilizados y el perfil explica el 25 % del ruido del residuo de T48.,"Suavizar con medias y pendientes móviles de varios vuelos e incluir descriptores del vuelo (duración, altitud, potencia)."
2,Cada motor arranca con una salud inicial distinta.,Restar a cada motor su propia línea base (primeros vuelos) y usar medias y pendientes móviles.
3,Vidas de 48 a 100 vuelos; los vuelos de un motor están correlacionados.,"Partir train / calibración / test por motor completo, nunca por vuelo."
4,"El 31 % de la vida es etapa sana, casi sin señal.",Esperar más incertidumbre lejos de la falla: intervalos que se adapten y métricas por tramo de RUL.
5,El score de NASA castiga más sobreestimar la vida.,Intervalos conformales asimétricos: menos tolerancia a quedar por encima del RUL real.
6,Los vuelos largos viven menos vuelos (70 vs 86).,Incluir la clase de vuelo y la duración como variables; evaluar la transferencia entre clases (Línea 2).
7,"El 99 % de los vuelos extremos está en la etapa degradada; entre los sanos sólo el 0,7 %.",No eliminar vuelos: los extremos son la señal que hay que predecir.
8,Los sensores virtuales y los parámetros de salud no existen en un avión real.,Usarlos sólo para entender el EDA; el modelo usa condiciones de vuelo y los 14 sensores físicos.


In [29]:
pd.DataFrame(E['takeaways'])

,title,text
0,"Datos limpios, con un archivo dañado","69,9 millones de registros sin nulos ni infinitos y con todas las verificaciones de consistencia en orden. El archivo de DS08d viene truncado en Kaggle y quedó afuera."
1,Los sensores reflejan sobre todo cómo se vuela,"Las cuatro condiciones de vuelo explican el 99,998 % de un sensor físico típico. En T48, el desgaste de toda la vida suma +29 °R, contra variaciones de ±123 °R dentro de un mismo vuelo."
2,Hay que descontar el efecto del vuelo,"Con un modelo de motor sano se obtienen residuos donde el desgaste se ve claro: en el motor de ejemplo, la correlación de T48 con el RUL pasa de -0,42 a -0,99."
3,Cada escenario de falla es distinto,"Hay 7 combinaciones distintas de componentes dañados en 9 subconjuntos, y cada una cambia qué sensores reaccionan."
4,El desgaste tiene dos etapas,"Primero lento y casi lineal, después cada vez más rápido. Los motores están sanos sólo el 31 % de su vida, en promedio."
5,El tipo de vuelo influye en la vida,"Los motores de vuelos largos duran 70 vuelos en promedio y los de vuelos cortos, 86."
